# Leveraging Geospatial Data in Python with GeoPandas

Real Barcelona district and bicycle data, projected area/distance, actual GeoJSON I/O, plots and two join types. Clone this repository and run from its root to access the included ten-row population CSV. Those aggregate totals are recovered from the original source table; no missing sex-level records are invented. Public bicycle data and map tiles require Internet access. No API key or paid service. All original code and twelve original screenshots remain in the nonexecuting appendix; freshly calculated tables/maps are displayed above it.

In [ ]:
import os, io, json, hashlib, tempfile
from pathlib import Path
from urllib.request import urlopen
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx
from shapely.geometry import Point
from pyproj import Geod, Transformer
from IPython.display import display, Image as NotebookImage
# python -m pip install geopandas==1.2.0 shapely==2.2.0 pyproj==3.7.2 pyogrio==0.13.0 pandas==2.3.3 matplotlib==3.11.2 contextily==1.7.1
# Alternative: conda install -c conda-forge geopandas contextily matplotlib
work = tempfile.TemporaryDirectory(prefix='tdb-geopandas-')
root = Path(work.name)
figures = Path(os.environ.get('TDB_FIGURE_OUTPUT',str(root/'figures')))
figures.mkdir(parents=True,exist_ok=True)
def save_figure(fig, number):
    target = figures/f'{number:02d}-current.png'
    fig.savefig(target,dpi=160,bbox_inches='tight',facecolor='white')
    plt.close(fig)
    display(NotebookImage(filename=str(target)))
def table_figure(frame,number):
    frame = frame.copy().astype(str)
    fig,ax = plt.subplots(figsize=(10,max(1.5,0.34*(len(frame)+1))))
    ax.axis('off')
    table=ax.table(cellText=frame.values,colLabels=frame.columns,cellLoc='left',loc='center')
    table.auto_set_font_size(False);table.set_fontsize(9);table.scale(1,1.4)
    for (row,column),cell in table.get_celld().items():
        cell.set_edgecolor('#d8e0ee')
        if row==0:cell.set_facecolor('#e9effb');cell.set_text_props(weight='bold')
    save_figure(fig,number)
print('GeoPandas:',gpd.__version__,'; actual Barcelona vector data and local spatial operations.')

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/jcanalesluna/bcn-geodata/da2a86de674aa5567cd5351e2940068bc7d546a6/districtes/districtes.geojson'
with urlopen(DATA_URL,timeout=60) as response:
    district_bytes = response.read()
districts = gpd.read_file(io.BytesIO(district_bytes))
assert len(districts)==10 and districts['DISTRICTE'].is_unique
assert districts.geometry.notna().all() and districts.geometry.is_valid.all()
assert districts.crs.to_epsg()==4326
print('Source SHA256:',hashlib.sha256(district_bytes).hexdigest())
print(districts[['DISTRICTE','NOM']].head().to_string(index=False))
table_figure(districts[['DISTRICTE','NOM']].head(),1)

In [ ]:
output = root/'districts.geojson'
districts.to_file(output,driver='GeoJSON')
restored = gpd.read_file(output)
assert restored.crs == districts.crs and restored['NOM'].tolist()==districts['NOM'].tolist()
assert restored.geometry.geom_equals(districts.geometry).all()
print('GeoJSON round trip:',len(restored),'districts; CRS, names and geometries preserved.')

In [ ]:
print('Input CRS:',districts.crs)
# Reproject coordinates; set_crs alone would only relabel them.
districts = districts.to_crs(epsg=25831)  # ETRS89 / UTM zone31N, metres, suitable for Barcelona.
assert districts.crs.is_projected and all(axis.unit_name=='metre' for axis in districts.crs.axis_info)
print('Analysis CRS:',districts.crs)
table_figure(pd.DataFrame({'Stage':['Input','Analysis'],'CRS':['EPSG:4326','EPSG:25831'],
    'Units':['degrees','metres']}),2)

In [ ]:
districts['area'] = districts.area/1_000_000  # square metres → square kilometres
assert districts['area'].between(1,30).all()
print(districts[['NOM','area']].round(6).to_string(index=False))
table_figure(districts[['NOM','area']].rename(columns={'area':'Area (km²)'}).round(6),3)

In [ ]:
districts['centroid'] = districts.centroid
assert districts['centroid'].crs == districts.crs
centroid_table = pd.DataFrame({'District':districts['NOM'],'X (m)':districts.centroid.x.round(3),
    'Y (m)':districts.centroid.y.round(3)})
print(centroid_table.to_string(index=False))
table_figure(centroid_table,4)

In [ ]:
districts['boundary'] = districts.boundary
assert districts['boundary'].notna().all()
boundary_table = pd.DataFrame({'District':districts['NOM'],'Boundary type':districts['boundary'].geom_type,
    'Boundary length (km)':(districts['boundary'].length/1000).round(4)})
print(boundary_table.to_string(index=False))
table_figure(boundary_table,5)

In [ ]:
church_lon, church_lat = 2.1743680500855005,41.403656946781304
sagrada_fam = gpd.GeoSeries([Point(church_lon,church_lat)],crs=4326).to_crs(districts.crs)
districts['sagrada_fam_dist'] = districts['centroid'].distance(sagrada_fam.iloc[0])/1000
assert districts['sagrada_fam_dist'].between(0,20).all()
print(districts[['NOM','sagrada_fam_dist']].round(6).to_string(index=False))
table_figure(districts[['NOM','sagrada_fam_dist']].rename(columns={'sagrada_fam_dist':'Distance (km)'}).round(6),6)

In [ ]:
ax = districts.plot(figsize=(10,6))
ax.set_title('Barcelona districts — EPSG:25831 (metres)')
ax.set_xlabel('Easting (m)');ax.set_ylabel('Northing (m)')
save_figure(ax.figure,7)
print('Rendered all ten projected district polygons.')

In [ ]:
ax = districts.plot(column='DISTRICTE',categorical=True,figsize=(10,6),edgecolor='black',legend=True)
ax.set_title('Barcelona districts by code')
save_figure(ax.figure,8)
print('Categorical district map rendered with all ten district codes.')

In [ ]:
ax = districts.plot(column='DISTRICTE',categorical=True,figsize=(12,6),alpha=0.5,legend=True)
districts['centroid'].plot(ax=ax,color='green')
# A bounded, attributed public tile request, not a fabricated local basemap.
web_bounds = districts.to_crs(3857).total_bounds
tiles = cx.howmany(*web_bounds,zoom=11,verbose=False)
assert tiles <= 16
import textwrap
provider = cx.providers.NASAGIBS.ASTER_GDEM_Greyscale_Shaded_Relief
cx.add_basemap(ax,crs=districts.crs.to_string(),source=provider,zoom=11,
    attribution=textwrap.fill(provider.attribution,82),attribution_size=6)
ax.set_title('Barcelona districts and centroids');ax.set_axis_off()
save_figure(ax.figure,9)
print('Actual NASA GIBS ASTER GDEM shaded-relief basemap; bounded tile count:',tiles)

In [ ]:
# Recover the ten visible original totals, not invented sex-level microdata.
repo_root = Path(os.environ.get('TDB_TUTORIAL_ROOT',str(Path.cwd())))
population_path = repo_root/'data/barcelona_2022_district_totals.csv'
if not population_path.exists():
    raise FileNotFoundError('Clone this tutorial repository and run from its root so data/barcelona_2022_district_totals.csv is available.')
pop = pd.read_csv(population_path)
assert len(pop)==10 and pop['district_code'].is_unique and pop['Nombre'].sum()==1_639_981
assert set(pop['district_code'])==set(range(1,11))
districts['district_code'] = districts['DISTRICTE'].astype(int)
name_check = districts[['district_code','NOM']].merge(pop,on='district_code',validate='one_to_one')
assert name_check['NOM'].tolist() == name_check['Nom_Districte'].tolist()
pop = pop.rename(columns={'Nombre':'population_22'})
districts = districts.merge(pop[['district_code','population_22']],on='district_code',validate='one_to_one')
assert len(districts)==10 and districts['population_22'].notna().all()
print(districts[['NOM','population_22']].to_string(index=False))
table_figure(districts[['NOM','population_22']],10)

In [ ]:
BIKE_URL = 'https://opendata-ajuntament.barcelona.cat/resources/bcn/CarrilsBici/CARRIL_BICI.geojson'
with urlopen(BIKE_URL,timeout=60) as response:
    bike_bytes = response.read()
bike_lane = gpd.read_file(io.BytesIO(bike_bytes))[['ID','geometry']].to_crs(districts.crs)
assert bike_lane.crs == districts.crs and len(bike_lane)>0
assert bike_lane['ID'].is_unique and bike_lane.geometry.notna().all() and bike_lane.geometry.is_valid.all()
print('Live public bicycle features:',len(bike_lane),'; SHA256:',hashlib.sha256(bike_bytes).hexdigest())
print(bike_lane[['ID']].head().to_string(index=False))
table_figure(pd.DataFrame({'Bike ID':bike_lane['ID'].head(),'Geometry':bike_lane.geometry.geom_type.head()}),11)

In [ ]:
lanes_districts = gpd.sjoin(districts,bike_lane,how='inner',predicate='intersects')
# Each row is a district/line match; crossing lines can match more than one district.
counts = lanes_districts.groupby('NOM')['ID'].nunique().rename('Intersecting bicycle features').reset_index()
assert len(counts)<=10 and len(lanes_districts)>0
print('District/line matches:',len(lanes_districts),'; distinct matched lines:',lanes_districts['ID'].nunique())
print(counts.to_string(index=False))
table_figure(counts,12)

In [ ]:
# Independent geodesic checks bound projected area/distance distortions.
geod = Geod(ellps='WGS84')
geographic = districts.to_crs(4326)
geodesic_areas = np.array([abs(geod.geometry_area_perimeter(geom)[0])/1e6 for geom in geographic.geometry])
np.testing.assert_allclose(districts['area'].to_numpy(),geodesic_areas,rtol=0.02,atol=0)
transformer = Transformer.from_crs(4326,25831,always_xy=True)
church_xy = np.array(transformer.transform(church_lon,church_lat))
centroid_xy = np.column_stack([districts['centroid'].x,districts['centroid'].y])
independent_km = np.linalg.norm(centroid_xy-church_xy,axis=1)/1000
np.testing.assert_allclose(districts['sagrada_fam_dist'],independent_km,atol=1e-10)
centroids_lonlat = gpd.GeoSeries(districts['centroid'],crs=25831).to_crs(4326)
geodesic_km = np.array([geod.inv(church_lon,church_lat,p.x,p.y)[2]/1000 for p in centroids_lonlat])
np.testing.assert_allclose(independent_km,geodesic_km,rtol=0.01,atol=0)
# Brute-force predicate checks validate the spatial-index join's actual pairs.
expected = {(int(row['district_code']),str(line['ID']))for _,row in districts.iterrows()
    for _,line in bike_lane.iterrows() if row.geometry.intersects(line.geometry)}
actual = set(zip(lanes_districts['district_code'].astype(int),lanes_districts['ID'].astype(str)))
assert expected==actual and len(actual)==len(lanes_districts)
assert len(list(figures.glob('*-current.png')))==12
print('Independent area/distance checks, exact attribute keys, all spatial pairs and twelve rendered figures passed.')
work.cleanup()

## Original source appendix — not executable

The original screenshots include inconsistent CRS measurements. They are retained as historical source material, not outputs of the maintained code above.

### Original block 1

```text
conda install geopandas
```

### Original block 2

```text
pip install geopandas
```

### Original block 3

```text
import geopandas as gpd
```

### Original block 4

```text
url = 'https://raw.githubusercontent.com/jcanalesluna/bcn-geodata/master/districtes/districtes.geojson'

districts = gpd.read_file(url)

districts.head()
```

### Original block 5

```text
districts.to_file("districts.geojson", driver="GeoJSON")
```

### Original block 6

```text
districts.crs
```

### Original block 7

```text
districts['area'] = districts.area / 1000000

districts['area']
```

### Original block 8

```text
districts['centroid']=districts.centroid

districts['centroid']
```

### Original block 9

```text
districts['boundary']=districts.boundary
```

### Original block 10

```text
from shapely.geometry import Point

sagrada_fam = Point(2.1743680500855005, 41.403656946781304)
sagrada_fam = gpd.GeoSeries(sagrada_fam, crs=4326)
sagrada_fam= sagrada_fam.to_crs(epsg=2062)
districts['sagrada_fam_dist'] = [float(sagrada_fam.distance(centroid)) / 1000 for centroid in districts.centroid]
```

### Original block 11

```text
ax= districts.plot(figsize=(10,6))
```

### Original block 12

```text
ax= districts.plot(column='DISTRICTE', figsize=(10,6), edgecolor='black', legend=True)
```

### Original block 13

```text
import contextily
import matplotlib.pyplot as plt

ax= districts.plot(column='DISTRICTE', figsize=(12,6), alpha=0.5, legend=True)
districts["centroid"].plot(ax=ax, color="green")
contextily.add_basemap(ax, crs=districts.crs.to_string())
plt.title('A Colored Map with the centroid of Barcelona')
plt.axis('off')
plt.show()
```

### Original block 14

```text
import pandas as pd
pop =pd.read_csv('2022_padro_sexe.csv', usecols=['Nom_Districte','Nombre'])
pop = pd.DataFrame(pop.groupby('Nom_Districte')['Nombre'].sum()).reset_index()
pop.columns=['NOM','population_22']
districts = districts.merge(pop)
districts
```

### Original block 15

```text
url = 'https://opendata-ajuntament.barcelona.cat/resources/bcn/CarrilsBici/CARRIL_BICI.geojson'
bike_lane = gpd.read_file(url)
bike_lane = bike_lane.loc[:,['ID','geometry']]
bike_lane.to_crs(epsg=2062, inplace=True)
```

### Original block 16

```text
lanes_districts = gpd.sjoin(districts, bike_lane, how='inner', predicate='intersects')
lanes_districts
```

### Original figure 1

![Original source figure 1](assets/geopandas-original/01-source.webp)

Source: https://cdn-images-1.medium.com/max/800/1*cP368wq69BAfIa-J4kUdsg.png

### Original figure 2

![Original source figure 2](assets/geopandas-original/02-source.webp)

Source: https://cdn-images-1.medium.com/max/800/1*TrgHbrGOXTp-O4xBuoevTg.png

### Original figure 3

![Original source figure 3](assets/geopandas-original/03-source.webp)

Source: https://cdn-images-1.medium.com/max/800/1*vds5-xtSNoxv47-9rgXtBA.png

### Original figure 4

![Original source figure 4](assets/geopandas-original/04-source.webp)

Source: https://cdn-images-1.medium.com/max/800/1*Yxb3A2soYIVkt0gvgfPTow.png

### Original figure 5

![Original source figure 5](assets/geopandas-original/05-source.webp)

Source: https://cdn-images-1.medium.com/max/800/1*QjhbPeza1HTkqP64GCm9Ig.png

### Original figure 6

![Original source figure 6](assets/geopandas-original/06-source.webp)

Source: https://cdn-images-1.medium.com/max/800/0*66w4mrcbn8t3fQgO.png

### Original figure 7

![Original source figure 7](assets/geopandas-original/07-source.webp)

Source: https://cdn-images-1.medium.com/max/800/1*I5Xyi4BHSXgIGXLljT-GWg.png

### Original figure 8

![Original source figure 8](assets/geopandas-original/08-source.webp)

Source: https://cdn-images-1.medium.com/max/800/1*Z2pnblcbcFStWzSSyy4kZg.png

### Original figure 9

![Original source figure 9](assets/geopandas-original/09-source.webp)

Source: https://cdn-images-1.medium.com/max/800/1*eePrytoNHAZEpycfn2UhtQ.png

### Original figure 10

![Original source figure 10](assets/geopandas-original/10-source.webp)

Source: https://cdn-images-1.medium.com/max/800/1*ZmxFQm3sb-qN_CJRGXfKwg.png

### Original figure 11

![Original source figure 11](assets/geopandas-original/11-source.webp)

Source: https://cdn-images-1.medium.com/max/800/0*hq0RLfZXb6wmSlrU.png

### Original figure 12

![Original source figure 12](assets/geopandas-original/12-source.webp)

Source: https://cdn-images-1.medium.com/max/800/0*EN9tuaQX5efql7QX.png